In [2]:
# Medicare Provider Analytics Using Apache Spark

## Notebook 3 – Data Integration

# Course:CS-675 Big Data Management & Analytics

# Author:Judi-Ann Beckford

### Objective

# Join the Medicare Provider Service dataset with the Medicare Enrollment dataset using the National Provider Identifier (NPI) to create a master analytics dataset.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [4]:
spark = (
    SparkSession.builder
    .appName("Medicare Data Integration")
    .config("spark.sql.shuffle.partitions", "24")
    .config("spark.default.parallelism", "24")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/29 19:12:02 WARN Utils: Your hostname, Judi-Anns-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.187 instead (on interface en0)
26/07/29 19:12:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
26/07/29 19:12:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2
Spark master: local[6]
Driver memory: 6g


In [5]:
provider_file = "../data/raw/PHY_R26_P05_V10_D24_Prov_Svc.csv"
enrollment_file = "../data/raw/PPEF_Enrollment_Extract_2026.04.01.csv"
output_path = "../data/processed/master_provider_services"

print("Provider file exists:", os.path.exists(provider_file))
print("Enrollment file exists:", os.path.exists(enrollment_file))
print("Current folder:", os.getcwd())

Provider file exists: True
Enrollment file exists: True
Current folder: /Users/judi-annbeckford/Documents/Medicare-Provider-Analytics-Spark/notebooks


In [6]:
provider_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(provider_file)
)

enrollment_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enrollment_file)
)

print("Provider-service columns:", len(provider_raw_df.columns))
print("Enrollment columns:", len(enrollment_raw_df.columns))

[Stage 3:==============================>                          (13 + 6) / 24]

Provider-service columns: 28
Enrollment columns: 11


In [7]:
provider_clean_df = (
    provider_raw_df
    .select(
        F.col("Rndrng_NPI").cast("long").alias("NPI"),
        F.trim(F.col("Rndrng_Prvdr_Type")).alias("PROVIDER_SPECIALTY"),
        F.upper(F.trim(F.col("Rndrng_Prvdr_State_Abrvtn"))).alias("PROVIDER_STATE"),
        F.trim(F.col("HCPCS_Cd")).alias("HCPCS_CODE"),
        F.trim(F.col("HCPCS_Desc")).alias("HCPCS_DESCRIPTION"),
        F.trim(F.col("Place_Of_Srvc")).alias("PLACE_OF_SERVICE"),
        F.col("Tot_Benes").cast("double").alias("TOTAL_BENEFICIARIES"),
        F.col("Tot_Srvcs").cast("double").alias("TOTAL_SERVICES"),
        F.col("Tot_Bene_Day_Srvcs").cast("double").alias("TOTAL_BENEFICIARY_DAY_SERVICES"),
        F.col("Avg_Sbmtd_Chrg").cast("double").alias("AVG_SUBMITTED_CHARGE"),
        F.col("Avg_Mdcr_Alowd_Amt").cast("double").alias("AVG_MEDICARE_ALLOWED"),
        F.col("Avg_Mdcr_Pymt_Amt").cast("double").alias("AVG_MEDICARE_PAYMENT"),
        F.col("Avg_Mdcr_Stdzd_Amt").cast("double").alias("AVG_STANDARDIZED_PAYMENT")
    )
    .filter(F.col("NPI").isNotNull())
    .filter(F.col("HCPCS_CODE").isNotNull())
)

In [8]:
provider_clean_df = (
    provider_clean_df
    .withColumn(
        "ESTIMATED_TOTAL_MEDICARE_PAYMENT",
        F.col("TOTAL_SERVICES") * F.col("AVG_MEDICARE_PAYMENT")
    )
    .withColumn(
        "ESTIMATED_TOTAL_SUBMITTED_CHARGE",
        F.col("TOTAL_SERVICES") * F.col("AVG_SUBMITTED_CHARGE")
    )
)

In [9]:
provider_clean_df.show(5, truncate=False)
provider_clean_df.printSchema()

[Stage 4:>                                                          (0 + 1) / 1]

+----------+------------------+--------------+----------+----------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+--------------+------------------------------+--------------------+--------------------+--------------------+------------------------+--------------------------------+--------------------------------+
|NPI       |PROVIDER_SPECIALTY|PROVIDER_STATE|HCPCS_CODE|HCPCS_DESCRIPTION                                                                                                                 |PLACE_OF_SERVICE|TOTAL_BENEFICIARIES|TOTAL_SERVICES|TOTAL_BENEFICIARY_DAY_SERVICES|AVG_SUBMITTED_CHARGE|AVG_MEDICARE_ALLOWED|AVG_MEDICARE_PAYMENT|AVG_STANDARDIZED_PAYMENT|ESTIMATED_TOTAL_MEDICARE_PAYMENT|ESTIMATED_TOTAL_SUBMITTED_CHARGE|
+----------+------------------+--------------+----------+-------------------------------------------------------------------------------------------

In [10]:
enrollment_selected_df = (
    enrollment_raw_df
    .select(
        F.col("NPI").cast("long").alias("NPI"),
        F.trim(F.col("PROVIDER_TYPE_DESC")).alias("ENROLLMENT_PROVIDER_TYPE"),
        F.upper(F.trim(F.col("STATE_CD"))).alias("ENROLLMENT_STATE"),
        F.trim(F.col("FIRST_NAME")).alias("FIRST_NAME"),
        F.trim(F.col("MDL_NAME")).alias("MIDDLE_NAME"),
        F.trim(F.col("LAST_NAME")).alias("LAST_NAME"),
        F.trim(F.col("ORG_NAME")).alias("ORGANIZATION_NAME")
    )
    .filter(F.col("NPI").isNotNull())
)

In [11]:
enrollment_dimension_df = (
    enrollment_selected_df
    .groupBy("NPI")
    .agg(
        F.first("ENROLLMENT_PROVIDER_TYPE", ignorenulls=True)
            .alias("ENROLLMENT_PROVIDER_TYPE"),
        F.first("ENROLLMENT_STATE", ignorenulls=True)
            .alias("ENROLLMENT_STATE"),
        F.first("FIRST_NAME", ignorenulls=True)
            .alias("FIRST_NAME"),
        F.first("MIDDLE_NAME", ignorenulls=True)
            .alias("MIDDLE_NAME"),
        F.first("LAST_NAME", ignorenulls=True)
            .alias("LAST_NAME"),
        F.first("ORGANIZATION_NAME", ignorenulls=True)
            .alias("ORGANIZATION_NAME"),
        F.count("*").alias("ENROLLMENT_RECORD_COUNT"),
        F.countDistinct("ENROLLMENT_PROVIDER_TYPE")
            .alias("DISTINCT_ENROLLMENT_TYPES"),
        F.countDistinct("ENROLLMENT_STATE")
            .alias("DISTINCT_ENROLLMENT_STATES")
    )
)

In [12]:
enrollment_dimension_rows = enrollment_dimension_df.count()
enrollment_dimension_unique_npis = (
    enrollment_dimension_df.select("NPI").distinct().count()
)

print(f"Enrollment dimension rows: {enrollment_dimension_rows:,}")
print(f"Unique NPIs: {enrollment_dimension_unique_npis:,}")
print(
    "NPI is unique:",
    enrollment_dimension_rows == enrollment_dimension_unique_npis
)

Enrollment dimension rows: 2,556,656
Unique NPIs: 2,556,656
NPI is unique: True


In [13]:
provider_rows_before_join = provider_clean_df.count()
provider_unique_npis = provider_clean_df.select("NPI").distinct().count()

print(f"Provider-service rows before join: {provider_rows_before_join:,}")
print(f"Provider-service unique NPIs: {provider_unique_npis:,}")

[Stage 20:==========================================>             (19 + 6) / 25]

Provider-service rows before join: 9,781,673
Provider-service unique NPIs: 1,207,473


In [14]:
master_df = (
    provider_clean_df
    .join(
        F.broadcast(enrollment_dimension_df),
        on="NPI",
        how="left"
    )
)

In [16]:
master_rows = master_df.count()

print(f"Rows before join: {provider_rows_before_join:,}")
print(f"Rows after join:  {master_rows:,}")
print("Row count preserved:", provider_rows_before_join == master_rows)

[Stage 26:========================================>               (18 + 6) / 25]

Rows before join: 9,781,673
Rows after join:  9,781,673
Row count preserved: True


In [17]:
matched_rows = master_df.filter(
    F.col("ENROLLMENT_PROVIDER_TYPE").isNotNull()
).count()

unmatched_rows = master_rows - matched_rows
match_rate = (matched_rows / master_rows) * 100

print(f"Matched service records:   {matched_rows:,}")
print(f"Unmatched service records: {unmatched_rows:,}")
print(f"Enrollment match rate:     {match_rate:.2f}%")

[Stage 33:============================>                           (12 + 6) / 24]

Matched service records:   9,645,673
Unmatched service records: 136,000
Enrollment match rate:     98.61%


In [18]:
master_df.select(
    "NPI",
    "PROVIDER_SPECIALTY",
    "PROVIDER_STATE",
    "ENROLLMENT_PROVIDER_TYPE",
    "ENROLLMENT_STATE",
    "HCPCS_CODE",
    "TOTAL_SERVICES",
    "AVG_MEDICARE_PAYMENT",
    "ESTIMATED_TOTAL_MEDICARE_PAYMENT"
).show(10, truncate=False)

[Stage 39:==========================================>             (18 + 6) / 24]

+----------+------------------+--------------+---------------------------------+----------------+----------+--------------+--------------------+--------------------------------+
|NPI       |PROVIDER_SPECIALTY|PROVIDER_STATE|ENROLLMENT_PROVIDER_TYPE         |ENROLLMENT_STATE|HCPCS_CODE|TOTAL_SERVICES|AVG_MEDICARE_PAYMENT|ESTIMATED_TOTAL_MEDICARE_PAYMENT|
+----------+------------------+--------------+---------------------------------+----------------+----------+--------------+--------------------+--------------------------------+
|1003000134|Pathology         |IL            |PRACTITIONER - PATHOLOGY         |IL              |88304     |56.0          |8.5091071429        |476.5100000024                  |
|1003000134|Pathology         |IL            |PRACTITIONER - PATHOLOGY         |IL              |88305     |1235.0        |25.907724696        |31996.03999956                  |
|1003000134|Pathology         |IL            |PRACTITIONER - PATHOLOGY         |IL              |88312     |85

In [19]:
state_comparison_df = (
    master_df
    .withColumn(
        "STATE_MATCH",
        F.when(
            F.col("PROVIDER_STATE") == F.col("ENROLLMENT_STATE"),
            "MATCH"
        )
        .when(F.col("ENROLLMENT_STATE").isNull(), "NO_ENROLLMENT_MATCH")
        .otherwise("DIFFERENT")
    )
)

state_comparison_df.groupBy("STATE_MATCH").count().show()

[Stage 50:================================>                       (14 + 6) / 24]

+-------------------+-------+
|        STATE_MATCH|  count|
+-------------------+-------+
|          DIFFERENT|1091586|
|              MATCH|8554087|
|NO_ENROLLMENT_MATCH| 136000|
+-------------------+-------+



In [20]:
(
    state_comparison_df
    .write
    .mode("overwrite")
    .partitionBy("PROVIDER_STATE")
    .parquet(output_path)
)

print("Master dataset saved to:", output_path)

Master dataset saved to: ../data/processed/master_provider_services


In [21]:
master_parquet_df = spark.read.parquet(output_path)

print(f"Saved Parquet rows: {master_parquet_df.count():,}")
print("Saved Parquet columns:", len(master_parquet_df.columns))

Saved Parquet rows: 9,781,673
Saved Parquet columns: 25


In [23]:
master_df.select(
    "NPI",
    "PROVIDER_SPECIALTY",
    "PROVIDER_STATE",
    "ENROLLMENT_PROVIDER_TYPE",
    "ENROLLMENT_STATE",
    "HCPCS_CODE",
    "TOTAL_SERVICES",
    "AVG_MEDICARE_PAYMENT",
    "ESTIMATED_TOTAL_MEDICARE_PAYMENT"
).show(10, truncate=False)

[Stage 68:============>   (18 + 6) / 24][Stage 69:>                (0 + 0) / 25]

+----------+------------------+--------------+---------------------------------+----------------+----------+--------------+--------------------+--------------------------------+
|NPI       |PROVIDER_SPECIALTY|PROVIDER_STATE|ENROLLMENT_PROVIDER_TYPE         |ENROLLMENT_STATE|HCPCS_CODE|TOTAL_SERVICES|AVG_MEDICARE_PAYMENT|ESTIMATED_TOTAL_MEDICARE_PAYMENT|
+----------+------------------+--------------+---------------------------------+----------------+----------+--------------+--------------------+--------------------------------+
|1003000134|Pathology         |IL            |PRACTITIONER - PATHOLOGY         |IL              |88304     |56.0          |8.5091071429        |476.5100000024                  |
|1003000134|Pathology         |IL            |PRACTITIONER - PATHOLOGY         |IL              |88305     |1235.0        |25.907724696        |31996.03999956                  |
|1003000134|Pathology         |IL            |PRACTITIONER - PATHOLOGY         |IL              |88312     |85

In [24]:
state_comparison_df = (
    master_df
    .withColumn(
        "STATE_MATCH",
        F.when(
            F.col("PROVIDER_STATE") == F.col("ENROLLMENT_STATE"),
            "MATCH"
        )
        .when(
            F.col("ENROLLMENT_STATE").isNull(),
            "NO_ENROLLMENT_MATCH"
        )
        .otherwise("DIFFERENT")
    )
)

state_comparison_df.groupBy("STATE_MATCH").count().show()

[Stage 80:==========================================>             (18 + 6) / 24]

+-------------------+-------+
|        STATE_MATCH|  count|
+-------------------+-------+
|          DIFFERENT|1091586|
|              MATCH|8554087|
|NO_ENROLLMENT_MATCH| 136000|
+-------------------+-------+



In [25]:
output_path = "../data/processed/master_provider_services"

(
    state_comparison_df
    .write
    .mode("overwrite")
    .partitionBy("PROVIDER_STATE")
    .parquet(output_path)
)

print("Master dataset successfully saved!")

Master dataset successfully saved!


In [22]:
## Integration Summary

#- Standardized the NPI field across both datasets.
#- Selected the variables required for provider utilization and payment analysis.
#- Created one enrollment dimension record per NPI to prevent row multiplication.
#- Used a left join to preserve all Medicare provider-service records.
#- Validated the join by comparing row counts before and after integration.
#- Calculated the enrollment matching rate.
#- Compared provider-service and enrollment state values.
#- Saved the integrated dataset in compressed, state-partitioned Parquet format.